In [2]:
import pandas as pd

In [3]:
subs_catboost = pd.read_csv("75.csv")
subs_tr_catboost = pd.read_csv("78282.csv")

subs_catboost["bin_flag"] = (subs_catboost["flag"] > 0.5).astype(int)
subs_tr_catboost["bin_flag"] = (subs_tr_catboost["flag"] > 0.5).astype(int)

In [4]:
subs_tr_catboost

,id,flag,bin_flag
0,4,0.438211,0
1,8,0.499621,0
2,10,0.270224,0
3,15,0.339351,0
4,16,0.097487,0
...,...,...,...
899995,2999962,0.487478,0
899996,2999965,0.784405,1
899997,2999980,0.801545,1
899998,2999983,0.115542,0


In [5]:
diff_idx = subs_catboost["bin_flag"] != subs_tr_catboost["bin_flag"]


diff_df = pd.DataFrame(
    {
        "id": subs_catboost.index[diff_idx],
        "flag_subs_catboost": subs_catboost.loc[diff_idx, "flag"],
        "bin_flag_subs_catboost": subs_catboost.loc[diff_idx, "bin_flag"],
        "flag_subs_tr_catboostr": subs_tr_catboost.loc[diff_idx, "flag"],
        "bin_flag_subs_tr_catboost": subs_tr_catboost.loc[diff_idx, "bin_flag"],
    }
)

diff_df.reset_index(drop=True, inplace=True)
diff_df

,id,flag_subs_catboost,bin_flag_subs_catboost,flag_subs_tr_catboostr,bin_flag_subs_tr_catboost
0,1,0.597532,1,0.499621,0
1,6,0.508790,1,0.468578,0
2,7,0.540994,1,0.498695,0
3,8,0.631566,1,0.453345,0
4,26,0.452442,0,0.572611,1
...,...,...,...,...,...
110619,899968,0.487940,0,0.676351,1
110620,899977,0.593962,1,0.484882,0
110621,899980,0.492349,0,0.579967,1
110622,899988,0.480314,0,0.614091,1


In [6]:
print(subs_catboost["flag"].min(), subs_catboost["flag"].max())
print(subs_tr_catboost["flag"].min(), subs_tr_catboost["flag"].max())

0.0135997953144017 0.9467634450271591
0.063234925 0.92533034


In [ ]:
# Псевдолейблинг

flag1 = subs_catboost["flag"]
flag2 = subs_tr_catboost["flag"]

two_ans = (flag1 > 0.5) == (flag2 > 0.5)
confident = ((flag1 > 0.91) & (flag2 > 0.91)) | ((flag1 < 0.09) & (flag2 < 0.09))
mask = two_ans & confident

pseudo_df = pd.DataFrame(
    {
        "id": subs_catboost.loc[mask, "id"]
        if "id" in subs_catboost.columns
        else subs_catboost.index[mask],
        "pseudo_target": ((flag1[mask] + flag2[mask]) / 2 > 0.5).astype(int),
        "pseudo_score_mean": ((flag1[mask] + flag2[mask]) / 2),
        "pseudo_conf": ((flag1[mask] + flag2[mask]) / 2 - 0.5).abs(),
    }
)
to_save_df = pseudo_df[["id", "pseudo_target"]]
to_save_df.to_csv("pseudo_labels.csv", index=False)
pseudo_df

,id,pseudo_target,pseudo_score_mean,pseudo_conf
99,291,0,0.065801,0.434199
154,442,0,0.082893,0.417107
183,543,0,0.069627,0.430373
195,597,0,0.060651,0.439349
205,623,0,0.071556,0.428444
...,...,...,...,...
899842,2999471,0,0.059867,0.440133
899890,2999619,0,0.077726,0.422274
899917,2999703,0,0.080996,0.419004
899922,2999716,0,0.086259,0.413741


In [12]:
t = pseudo_df[pseudo_df["pseudo_target"] == 1]
to_save_df = pseudo_df[["id", "pseudo_target"]]
to_save_df.to_csv("pseudo_labels.csv", index=False)

In [11]:
train_target = pd.read_csv("data\\train_target.csv")
train_target["flag"].value_counts()

flag
0    2025491
1      74509
Name: count, dtype: int64

In [ ]:
pseudo_df["pseudo_target"]

In [25]:
analysis_df = pd.DataFrame(
    {
        "id": subs_catboost["id"] if "id" in subs_catboost.columns else subs_catboost.index,
        "p_cat": subs_catboost["flag"],
        "p_tr_cat": subs_tr_catboost["flag"],
    }
)

analysis_df["abs_diff"] = (analysis_df["p_cat"] - analysis_df["p_tr_cat"]).abs()
analysis_df["mean_pred"] = (analysis_df["p_cat"] + analysis_df["p_tr_cat"]) / 2
analysis_df["bin_cat"] = (analysis_df["p_cat"] > 0.5).astype(int)
analysis_df["bin_tr_cat"] = (analysis_df["p_tr_cat"] > 0.5).astype(int)
analysis_df["bin_diff"] = analysis_df["bin_cat"] != analysis_df["bin_tr_cat"]

hard_disagreement = analysis_df.sort_values("abs_diff", ascending=False)

uncertain = analysis_df[
    (analysis_df["p_cat"].between(0.4, 0.6)) & (analysis_df["p_tr_cat"].between(0.4, 0.6))
].copy()

uncertain

,id,p_cat,p_tr_cat,abs_diff,mean_pred,bin_cat,bin_tr_cat,bin_diff
0,4,0.438153,0.438211,0.000058,0.438182,0,0,False
1,8,0.597532,0.499621,0.097911,0.548577,1,0,True
6,20,0.508790,0.468578,0.040212,0.488684,1,0,True
7,22,0.540994,0.498695,0.042299,0.519845,1,0,True
9,26,0.419251,0.433406,0.014154,0.426329,0,0,False
...,...,...,...,...,...,...,...,...
899965,2999873,0.529787,0.573511,0.043724,0.551649,1,1,False
899977,2999918,0.593962,0.484882,0.109080,0.539422,1,0,True
899979,2999923,0.470192,0.465726,0.004466,0.467959,0,0,False
899980,2999925,0.492349,0.579967,0.087618,0.536158,0,1,True


In [29]:
cat_conf_tr_uncertain = analysis_df[
    ((analysis_df["p_cat"] < 0.1) | (analysis_df["p_cat"] > 0.9))
    & (analysis_df["p_tr_cat"].between(0.4, 0.6))
]
cat_conf_tr_uncertain

,id,p_cat,p_tr_cat,abs_diff,mean_pred,bin_cat,bin_tr_cat,bin_diff
27239,89698,0.094371,0.412625,0.318254,0.253498,0,0,False
107228,356760,0.083647,0.405958,0.322310,0.244803,0,0,False
218261,728456,0.091464,0.454186,0.362722,0.272825,0,0,False
359172,1198406,0.098532,0.481576,0.383044,0.290054,0,0,False
378146,1261961,0.055530,0.494653,0.439123,0.275091,0,0,False
468391,1563122,0.094501,0.495054,0.400553,0.294778,0,0,False
622878,2076309,0.085423,0.408598,0.323175,0.247011,0,0,False
651436,2171653,0.095288,0.418661,0.323373,0.256975,0,0,False
677145,2257080,0.099466,0.407349,0.307883,0.253408,0,0,False
711697,2372010,0.092752,0.443439,0.350688,0.268095,0,0,False


In [34]:
tr_conf_cat_uncertain = analysis_df[
    ((analysis_df["p_tr_cat"] < 0.1) | (analysis_df["p_tr_cat"] > 0.9))
    & (analysis_df["p_cat"].between(0.4, 0.6))
]
tr_conf_cat_uncertain

,id,p_cat,p_tr_cat,abs_diff,mean_pred,bin_cat,bin_tr_cat,bin_diff
3869,12768,0.474932,0.092882,0.382050,0.283907,0,0,False
4138,13611,0.534799,0.092561,0.442238,0.313680,1,0,True
16918,55937,0.434034,0.083670,0.350364,0.258852,0,0,False
17779,58697,0.425950,0.082973,0.342977,0.254462,0,0,False
18799,62128,0.425520,0.083992,0.341528,0.254756,0,0,False
...,...,...,...,...,...,...,...,...
876768,2922800,0.468429,0.091170,0.377259,0.279799,0,0,False
883810,2946277,0.433732,0.097310,0.336422,0.265521,0,0,False
884703,2949184,0.405336,0.096110,0.309225,0.250723,0,0,False
888780,2962994,0.491274,0.089074,0.402200,0.290174,0,0,False


In [ ]:
flag1 = subs_catboost["flag"]
flag2 = subs_tr_catboost["flag"]

two_ans = (flag1 > 0.5) == (flag2 > 0.5)
confident = (flag1 < 0.1) & (flag2 < 0.1)
mask = two_ans & confident

pseudo_df = pd.DataFrame(
    {
        "id": subs_catboost.loc[mask, "id"]
        if "id" in subs_catboost.columns
        else subs_catboost.index[mask],
        "pseudo_target": ((flag1[mask] + flag2[mask]) / 2 > 0.5).astype(int),
        "pseudo_score_mean": ((flag1[mask] + flag2[mask]) / 2),
        "pseudo_conf": ((flag1[mask] + flag2[mask]) / 2 - 0.5).abs(),
    }
)
to_save_df = pseudo_df[["id", "pseudo_target"]]
to_save_df.to_csv("pseudo_labels.csv", index=False)
pseudo_df

лучший бленд

In [ ]:
xlstm_pred = pd.read_csv("xlstm_pred_test.csv")
bilstm_catboost = pd.read_csv("bilstm_catboost_seed43_test.csv")
transformer_bilstm_cross_attention_mlp = pd.read_csv(
    "transformer_bilstm_cross_attention_mlp_seed42_test.csv"
)

In [4]:
bilstm_catboost.rename(columns={"prediction": "flag"}, inplace=True)
bilstm_catboost.drop(columns="model", inplace=True)
bilstm_catboost.head()

,id,flag
0,4,0.107876
1,8,0.126014
2,10,0.141264
3,15,0.074424
4,16,0.013809


In [7]:
transformer_bilstm_cross_attention_mlp.rename(columns={"prediction": "flag"}, inplace=True)
transformer_bilstm_cross_attention_mlp.drop(columns="model", inplace=True)
transformer_bilstm_cross_attention_mlp.head()

,id,flag
0,4,0.133366
1,8,0.191830
2,10,0.107656
3,15,0.141002
4,16,0.024740


In [8]:
xlstm_pred.rename(columns={"pred_tr_cat": "flag"}, inplace=True)
xlstm_pred.head()

,id,flag
0,4,0.111563
1,8,0.136955
2,10,0.123056
3,15,0.076517
4,16,0.020241


In [10]:
res_subs = xlstm_pred.copy()

res_subs["flag"] = (
    bilstm_catboost["flag"] + transformer_bilstm_cross_attention_mlp["flag"] + xlstm_pred["flag"]
) / 3
res_subs.to_csv(
    "bilstm_catboost_transformer_bilstm_cross_attention_mlp_xlstm_pred.csv",
    index=False,
    float_format="%.5f",
)
res_subs

,id,flag
0,4,0.117602
1,8,0.151600
2,10,0.123992
3,15,0.097314
4,16,0.019597
...,...,...
899995,2999962,0.215991
899996,2999965,0.441360
899997,2999980,0.475448
899998,2999983,0.032312
